# Whisper IT 파인튜닝
L40S 46GB GPU에서 LoRA 파인튜닝

In [1]:
# [필수] 필요한 라이브러리 설치 (가장 먼저 한 번만 실행하세요)
!pip install -U pip
!pip install -U transformers datasets accelerate peft evaluate jiwer librosa soundfile ctranslate2

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
GPU_ID = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["HF_AUDIO_BACKEND"] = "soundfile"

import json
import torch
import numpy as np
import librosa
import soundfile as sf
from datasets import Dataset
from dataclasses import dataclass
from typing import Any, Dict, List
from transformers import (
    WhisperProcessor, 
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import evaluate

# 기본 설정값
MODEL_NAME = "openai/whisper-large-v3"
MAX_STEPS = 3000
BATCH_SIZE = 16 
BASE_DIR = os.path.expanduser("~/whisper_data")
METADATA_DIR = os.path.join(BASE_DIR, "whisper_train_data")
OUTPUT_DIR = os.path.join(BASE_DIR, "whisper-it-finetuned")

# 모델 및 프로세서 로드
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="ko", task="transcribe")
print("1단계 완료: 모델 로드 성공")

Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

1단계 완료: 모델 로드 성공


In [ ]:
@dataclass
class WhisperDataCollator:
    processor: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = []
        label_features = []
        
        # 한국어 transcribe prefix 토큰 가져오기
        forced_decoder_ids = self.processor.get_decoder_prompt_ids(language="ko", task="transcribe")
        prefix_tokens = [token_id for _, token_id in forced_decoder_ids]
        
        for f in features:
            audio, sr = sf.read(f["audio_path"])
            if len(audio.shape) > 1: audio = audio.mean(axis=1)
            if sr != 16000: audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
            feat = self.processor.feature_extractor(audio, sampling_rate=16000).input_features[0]
            input_features.append({"input_features": feat})
            
            # 텍스트 토큰화 (prefix 없이)
            text_tokens = self.processor.tokenizer(f["sentence"], add_special_tokens=False).input_ids
            
            # prefix + 텍스트 + EOS 조합
            full_tokens = prefix_tokens + text_tokens + [self.processor.tokenizer.eos_token_id]
            label_features.append({"input_ids": full_tokens})

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

class WhisperPeftTrainer(Seq2SeqTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model.base_model.model(
            input_features=inputs.get("input_features"),
            labels=inputs.get("labels")
        )
        return (outputs.loss, outputs) if return_outputs else outputs.loss

print("2단계 완료: 설계도 정의 성공")

In [4]:
# 1. 데이터셋 로드
def load_ds(path):
    with open(path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    return Dataset.from_dict({
        "audio_path": [os.path.join(BASE_DIR, i["audio"]) for i in metadata],
        "sentence": [i["text"] for i in metadata]
    })

train_ds = load_ds(os.path.join(METADATA_DIR, "metadata_train.json"))
val_ds = load_ds(os.path.join(METADATA_DIR, "metadata_val.json"))

# 2. LoRA 설정
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], task_type="SEQ_2_SEQ_LM"))

# 3. 객체 생성 (순서: Collator -> Args -> Trainer)
data_collator = WhisperDataCollator(processor=processor) # 에러 해결 포인트

training_args = Seq2SeqTrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "checkpoints"),
    per_device_train_batch_size=8,       # 16에서 8로 하향
    gradient_accumulation_steps=2,       # 8 * 2 = 16 효과 유지
    learning_rate=1e-4,
    warmup_steps=100,
    max_steps=MAX_STEPS,
    fp16=True,                           # fp16은 반드시 유지 (메모리 절약)
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=50,
    predict_with_generate=True,          # 평가 시 메모리 많이 먹는 주범
    per_device_eval_batch_size=4,        # 평가 배치는 더 낮게 설정 (중요!)
    generation_max_length=225,
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer = WhisperPeftTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=None # 속도를 위해 일단 None으로 진행 가능
)

print("3단계: 모든 준비 완료. 학습 시작!")
trainer.train()

3단계: 모든 준비 완료. 학습 시작!


Step,Training Loss,Validation Loss
500,0.159616,0.082366
1000,0.101678,0.047960
1500,0.068478,0.037247
2000,0.059972,0.032156
2500,0.055488,0.028137
3000,0.042587,0.026934


TrainOutput(global_step=3000, training_loss=0.1596956020196279, metrics={'train_runtime': 12874.8771, 'train_samples_per_second': 3.728, 'train_steps_per_second': 0.233, 'total_flos': 1.6387450600292352e+20, 'train_loss': 0.1596956020196279, 'epoch': 2.223952539859103})

In [ ]:
# 1. LoRA 어댑터 가중치 저장
lora_path = os.path.join(OUTPUT_DIR, "whisper-it-lora")
model.save_pretrained(lora_path)
print(f"LoRA 가중치 저장 완료: {lora_path}")

# 2. 모델 병합 (Base Model + LoRA)
print("모델 병합 중...")
merged_model = model.merge_and_unload()

merged_path = os.path.join(OUTPUT_DIR, "whisper-it-merged")
merged_model.save_pretrained(merged_path)
processor.save_pretrained(merged_path)
print(f"병합 모델 저장 완료: {merged_path}")

# 3. CTranslate2 (CT2) 변환 - Python API 사용
import ctranslate2
import shutil

ct2_path = os.path.join(OUTPUT_DIR, "whisper-it-ct2")
print(f"CT2 양자화 변환 시작 (float16)...")

converter = ctranslate2.converters.TransformersConverter(merged_path)
converter.convert(ct2_path, quantization="float16")

# preprocessor_config.json 복사 (large-v3용 128 mel bins)
src_config = os.path.join(merged_path, "preprocessor_config.json")
dst_config = os.path.join(ct2_path, "preprocessor_config.json")
if os.path.exists(src_config):
    shutil.copy(src_config, dst_config)
    print(f"preprocessor_config.json 복사 완료")

print(f"✅ CT2 변환 완료! 경로: {ct2_path}")